# Stage 2 - Clean and align the data

Builds the monthly datasets used by the next stages. Reads `data/raw/` (never changed) and writes to `data/processed/`:

- `macro_realtime_12m.csv`, `macro_realtime_3m.csv`: macro indicators as an investor could see them at each month-end (ALFRED vintages), as 12-month and 3-month changes
- `macro_revised_12m.csv`, `macro_revised_3m.csv`: the same indicators with today's revised values, same timing
- `macro_data_month.csv`: which month of data was the latest available at each month-end
- `returns_daily.csv`: daily log returns of the 4 ETFs
- `returns_monthly.csv`: monthly simple returns of the 4 ETFs
- `stock_bond_corr.csv`: SPY-TLT correlation of daily returns, one value per month

Run from top to bottom in a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd

import config
import helpers

## 1. Load the raw files

In [ ]:
vintages = helpers.load_alfred_vintages()
revised = helpers.load_revised_macro()
prices = helpers.load_etf_prices("Adj Close")

print("vintages:", vintages.shape, "| revised:", revised.shape, "| prices:", prices.shape)
print("missing values in vintages:", vintages["value"].isna().sum())
print(vintages[vintages["value"].isna()][["series_id", "date", "realtime_start"]])

## 2. Choices

- **Transformations.** Levels trend over time, so each indicator becomes a change, computed two ways:
  over 12 months (main version: smooth, shows the trend) and over 3 months, annualised (robustness check: faster but noisier).
  Log change in % for indices and quantities, change in points for the unemployment rate,
  level for capacity utilisation (already a rate that moves around a mean).
- **Initial claims (ICSA)** are weekly: they become the monthly average of the weeks, complete months only.
  ALFRED has their vintages only from May 2009; before that the revised values are used.
  Claims are almost never revised (median revision of the 12-month change after 2009: about 0.5 points), so this changes little.
- **Timing.** A decision is taken at each month-end. Only data published by that day can be used.
- **Missing months** (October 2025, shutdown): the last available value is kept, as an investor would.
- **Sample.** Decision dates from January 2001 (12-month changes need 2000) to August 2026,
  the last complete month before the freeze date (28 September 2026).

In [ ]:
# indicator -> (column name, transformation)
TRANSFORMS = {
    "INDPRO":   ("indpro",   "log"),
    "TCU":      ("tcu",      "level"),
    "RRSFS":    ("retail",   "log"),
    "HOUST":    ("housing",  "log"),
    "PAYEMS":   ("payrolls", "log"),
    "UNRATE":   ("unrate",   "diff"),
    "ICSA":     ("claims",   "log"),
    "CPIAUCSL": ("cpi",      "log"),
    "CPILFESL": ("core_cpi", "log"),
    "PPIACO":   ("ppi",      "log"),
}
HORIZONS = [12, 3]   # months
FIRST_ICSA_VINTAGE = vintages.loc[vintages["series_id"] == "ICSA", "realtime_start"].min()

decision_dates = pd.date_range("2001-01-31", "2026-08-31", freq="ME")
print(len(decision_dates), "decision dates:", decision_dates[0].date(), "->", decision_dates[-1].date())
print("first ICSA vintage:", FIRST_ICSA_VINTAGE)

In [ ]:
def transform(s, how, h):
    """Change over h months: log change in % (annualised), change in points, or level."""
    if how == "log":
        return 100 * (12 / h) * np.log(s / s.shift(h))
    if how == "diff":
        return s - s.shift(h)
    return s


def to_monthly(values, sid, t=None):
    """Monthly series indexed by the first day of the month. ICSA: average of the weeks, complete months only."""
    values = values.copy()
    values.index = pd.to_datetime(values.index)
    if sid != "ICSA":
        return values.asfreq("MS")
    monthly = values.resample("MS").mean()
    if t is not None:                                    # the month of t is not complete yet
        monthly = monthly[monthly.index < t.to_period("M").to_timestamp()]
    return monthly

## 3. Real-time panels

For each decision date `t` and each indicator:
1. keep the vintage that was valid on `t` (`realtime_start <= t <= realtime_end`): the data as published that day;
2. apply the transformations to that vintage;
3. take the latest month with a value, and save which month it is.

ICSA before its first vintage uses the revised data, cut at the month before `t`.

In [ ]:
revised_monthly = {sid: to_monthly(revised[revised["series_id"] == sid].set_index("date")["value"], sid)
                   for sid in TRANSFORMS}


def realtime_values(vint, sid, how, t):
    """Latest transformed values of one indicator, using only data published by date t."""
    ts = t.strftime("%Y-%m-%d")
    if sid == "ICSA" and ts < FIRST_ICSA_VINTAGE:
        s = revised_monthly["ICSA"]
        s = s[s.index < t.to_period("M").to_timestamp()]
    else:
        snap = vint[(vint["realtime_start"] <= ts) & (vint["realtime_end"] >= ts)]
        if snap.empty:                                   # nothing published yet (RRSFS before June 2001)
            return {h: np.nan for h in HORIZONS}, pd.NaT
        s = to_monthly(snap.set_index("date")["value"], sid, t)

    x12 = transform(s, how, 12).dropna()
    if x12.empty:
        return {h: np.nan for h in HORIZONS}, pd.NaT
    month = x12.index[-1]                                # latest month with a value (skips October 2025)
    # ffill: a change that needs October 2025 keeps the last value, as in the revised panel
    return {h: transform(s, how, h).ffill().get(month, np.nan) for h in HORIZONS}, month


rt = {h: {} for h in HORIZONS}
months = {}
for sid, (name, how) in TRANSFORMS.items():
    vint = vintages[vintages["series_id"] == sid]
    out = [realtime_values(vint, sid, how, t) for t in decision_dates]
    for h in HORIZONS:
        rt[h][name] = [vals[h] for vals, m in out]
    months[name] = [m for vals, m in out]

macro_rt = {h: pd.DataFrame(rt[h], index=decision_dates).rename_axis("date") for h in HORIZONS}
data_month = pd.DataFrame(months, index=decision_dates).rename_axis("date")
macro_rt[12].tail()

In [ ]:
# how old is the information at each decision date (months between data month and decision date)
age = pd.DataFrame({c: (decision_dates.year - data_month[c].dt.year) * 12
                       + (decision_dates.month - data_month[c].dt.month) for c in data_month},
                   index=decision_dates)
print("data age in months (median / max):")
print(age.agg(["median", "max"]).T)

## 4. Revised panels, same timing

Same transformations on today's revised data, read at the **same data month** as the real-time panel.
So the two panels differ only because of revisions, not because of timing.

In [ ]:
macro_rev = {}
for h in HORIZONS:
    cols = {}
    for sid, (name, how) in TRANSFORMS.items():
        x = transform(revised_monthly[sid], how, h).ffill()   # October 2025 gap: keep the last value
        cols[name] = x.reindex(data_month[name]).to_numpy()
    macro_rev[h] = pd.DataFrame(cols, index=decision_dates).rename_axis("date")

for h in HORIZONS:
    print(f"\n{h}-month changes, mean absolute difference real time vs revised (from 2006):")
    print((macro_rt[h] - macro_rev[h]).loc["2006":].abs().mean().round(2).to_string())

## 5. ETF returns

- Daily log returns from Adjusted Close (dividends included), kept for covariance estimates later.
- Monthly simple returns from the last price of each month.
- DBC starts in February 2006, so all 4 assets have returns from March 2006.
- September 2026 is incomplete (data stop on the 28th) and is dropped.

In [ ]:
last_month_end = decision_dates[-1]

ret_daily = np.log(prices).diff().loc[:last_month_end]
ret_daily = ret_daily.dropna(how="all")

month_prices = prices.resample("ME").last().loc[:last_month_end]
ret_monthly = month_prices.pct_change(fill_method=None).dropna(how="all")

common = ret_monthly.dropna().index
print("monthly returns, all 4 assets:", common[0].date(), "->", common[-1].date(), f"({len(common)} months)")
ret_monthly.describe().round(4)

## 6. Stock-bond correlation, one value per month

Correlation of daily SPY and TLT returns inside each month (about 21 days).
This is the variable the regimes should explain.
Months with fewer than 15 days of both returns are left out (TLT starts at the end of July 2002).

In [ ]:
def month_corr(g):
    g = g[["SPY", "TLT"]].dropna()
    return g["SPY"].corr(g["TLT"]) if len(g) >= 15 else np.nan


sb_corr = ret_daily.groupby(ret_daily.index.to_period("M")).apply(month_corr).dropna()
sb_corr.index = sb_corr.index.to_timestamp("M")
sb_corr = sb_corr.rename("spy_tlt_corr").to_frame()
sb_corr.index.name = "date"

print("average correlation by year:")
print(sb_corr["spy_tlt_corr"].groupby(sb_corr.index.year).mean().round(2).to_string())

## 7. Checks and save

In [ ]:
# the macro panels have the same dates, and every month with returns has a decision date
for h in HORIZONS:
    assert macro_rt[h].index.equals(macro_rev[h].index)
assert ret_monthly.loc["2001":].index.isin(decision_dates).all()
assert sb_corr.index.isin(decision_dates).all()

# no missing macro values from 2006 on (the asset sample)
for h in HORIZONS:
    print(f"missing values from 2006, {h}m: real time {int(macro_rt[h].loc['2006':].isna().sum().sum())},"
          f" revised {int(macro_rev[h].loc['2006':].isna().sum().sum())}")

outputs = {"macro_realtime_12m": macro_rt[12], "macro_revised_12m": macro_rev[12],
           "macro_realtime_3m": macro_rt[3], "macro_revised_3m": macro_rev[3],
           "macro_data_month": data_month, "returns_daily": ret_daily,
           "returns_monthly": ret_monthly, "stock_bond_corr": sb_corr}
for name, df in outputs.items():
    path = helpers.save_processed(df, name)
    print(f"saved {path.name}: {df.shape[0]} rows x {df.shape[1]} columns")